# EvoVariant-TR autonomous adaptation

Free Colab GPU only. Selection uses TRAIN grouped CV/OOF; VALIDATION remains closed until a hash-bound selection lock exists. The 946-row locked cohort is never loaded.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
ROOT = Path('/content/EvoVariant')
DRIVE_ROOT = Path('/content/drive/MyDrive/EvoVariantTR')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
for name in ('reference','datasets','checkpoints','model_cache','runs','hpo','logs','state','exports'):
    (DRIVE_ROOT / name).mkdir(exist_ok=True)


In [ ]:
BRANCH = 'research/posthoc-foundation-adaptation'
if not ROOT.exists():
    subprocess.run(['git','clone','-b',BRANCH,'https://github.com/UtkarsHMer05/EvoVariant-TR-.git',str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'pull','--ff-only','origin',BRANCH],check=True)
print(subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())
print(subprocess.check_output(['git','-C',str(ROOT),'status','--short','--branch'],text=True).strip())


In [ ]:
ENV = Path('/content/caduceus-env')
UV = shutil.which('uv') or '/usr/local/bin/uv'
if not shutil.which('uv') and not Path(UV).exists():
    subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
    UV = shutil.which('uv') or '/usr/local/bin/uv'
if not ENV.exists():
    subprocess.run([UV,'venv','--python','3.11',str(ENV)],check=True)
PY = str(ENV / 'bin' / 'python')
subprocess.run([UV,'pip','install','--python',PY,'--index-url','https://download.pytorch.org/whl/cu121','torch==2.2.0'],check=True)
subprocess.run([UV,'pip','install','--python',PY,'--no-build-isolation','-r',str(ROOT/'requirements-adaptation.txt')],check=True)
print(subprocess.check_output([PY,'-c','import torch,transformers,mamba_ssm; print(torch.__version__,torch.cuda.is_available(),torch.cuda.get_device_name(0),transformers.__version__)'],text=True))


In [ ]:
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
STATE = DRIVE_ROOT / 'state' / 'adaptation_state.json'
subprocess.run([PY,'scripts/adaptation/hardware_probe.py','--output',str(DRIVE_ROOT/'state'/'environment.json'),'--state',str(STATE),'--root',str(ROOT)],check=True)
subprocess.run([PY,'scripts/adaptation/verify_data.py','--root',str(ROOT),'--output',str(DRIVE_ROOT/'state'/'manifests_verified.json'),'--state',str(STATE)],check=True)


In [ ]:
import gzip, hashlib, shutil
ARCHIVE = DRIVE_ROOT / 'reference' / 'hg38.fa.gz'
REFERENCE = Path('/content/Homo_sapiens_assembly38.fasta')
EXPECTED_ARCHIVE_BYTES = 983659424
EXPECTED_REFERENCE_SHA256 = '5be01555d98347fdb3714dc84c6f77c9d8bc774adcf32c6f7a8fa06f5baf5e51'
if ARCHIVE.stat().st_size != EXPECTED_ARCHIVE_BYTES:
    raise RuntimeError(f'unexpected UCSC hg38 archive size: {ARCHIVE.stat().st_size}')
digest = hashlib.sha256()
with gzip.open(ARCHIVE,'rb') as source, REFERENCE.open('wb') as target:
    while chunk := source.read(8*1024*1024):
        target.write(chunk)
        digest.update(chunk)
actual = digest.hexdigest()
if actual != EXPECTED_REFERENCE_SHA256:
    raise RuntimeError(f'UCSC hg38 FASTA SHA-256 mismatch: {actual}')
subprocess.run([PY,'-c',f'from pyfaidx import Fasta; Fasta({str(REFERENCE)!r}, as_raw=True, sequence_always_upper=True)'],check=True)
shutil.copy2(REFERENCE.with_suffix('.fasta.fai'),DRIVE_ROOT/'reference'/'Homo_sapiens_assembly38.fasta.fai')
subprocess.run([PY,'scripts/adaptation/verify_data.py','--root',str(ROOT),'--reference',str(REFERENCE),'--output',str(DRIVE_ROOT/'state'/'data_ready.json'),'--state',str(STATE)],check=True)
print('reference',REFERENCE,'bytes',REFERENCE.stat().st_size,'sha256',actual)
print('drive_archive_bytes',ARCHIVE.stat().st_size,'fai_bytes',(DRIVE_ROOT/'reference'/'Homo_sapiens_assembly38.fasta.fai').stat().st_size)


In [ ]:
SMOKE = DRIVE_ROOT/'runs'/'caduceus_smoke.json'
SMOKE_CKPT = DRIVE_ROOT/'checkpoints'/'caduceus_smoke.pt'
subprocess.run([PY,'scripts/adaptation/smoke_caduceus.py','--device','cuda','--cache-dir',str(DRIVE_ROOT/'model_cache'),'--output',str(SMOKE),'--checkpoint',str(SMOKE_CKPT),'--state',str(STATE),'--root',str(ROOT)],check=True)


In [ ]:
HPO = DRIVE_ROOT/'hpo'/'caduceus_hpo.json'
subprocess.run([PY,'scripts/adaptation/run_caduceus_hpo.py','--root',str(ROOT),'--reference',str(REFERENCE),'--cache-dir',str(DRIVE_ROOT/'model_cache'),'--checkpoint-dir',str(DRIVE_ROOT/'checkpoints'/'caduceus_hpo'),'--state',str(STATE),'--output',str(HPO),'--device','cuda','--trials','8'],check=True)


In [ ]:
LOCK = DRIVE_ROOT/'hpo'/'selection_closed.json'
if not LOCK.is_file():
    raise RuntimeError('TRAIN-only HPO is incomplete; holdout stays closed')
selection = __import__('json').loads(LOCK.read_text())
params = selection['selected_params']
RUN = DRIVE_ROOT/'runs'/'caduceus_final'
cmd = [PY,'scripts/adaptation/train_caduceus.py','--root',str(ROOT),'--reference',str(REFERENCE),'--cache-dir',str(DRIVE_ROOT/'model_cache'),'--output-dir',str(RUN),'--state',str(STATE),'--selection-lock',str(LOCK),'--device','cuda','--stage',params['regime'],'--epochs',str(selection['final_epochs']),'--batch-size','1','--effective-batch-size',str(params['effective_batch_size']),'--dropout',str(params['dropout']),'--learning-rate',str(params['learning_rate']),'--weight-decay',str(params['weight_decay']),'--seed',str(selection['seed'])]
if (RUN/'latest.pt').exists(): cmd += ['--resume',str(RUN/'latest.pt')]
subprocess.run(cmd,check=True)


In [ ]:
HOLDOUT = DRIVE_ROOT/'exports'/'caduceus_validation.csv'
subprocess.run([PY,'scripts/adaptation/evaluate_caduceus.py','--root',str(ROOT),'--reference',str(REFERENCE),'--cache-dir',str(DRIVE_ROOT/'model_cache'),'--checkpoint',str(RUN/'latest.pt'),'--selection-lock',str(LOCK),'--state',str(STATE),'--output',str(HOLDOUT),'--split','validation','--device','cuda'],check=True)
